# MLOps Story — Pourquoi tout ce travail ?

Ce notebook illustre visuellement la valeur du pipeline MLOps :
- Comparaison des modèles (GridSearch)
- Impact du réentraînement sur les performances
- Stabilité des prédictions
- Dérive simulée des données

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('Setup OK')

In [ ]:
TARGET = 'listening_time'
CATEGORICAL = ['gender', 'country', 'subscription_type', 'device_type']
NUMERICAL = ['age', 'songs_played_per_day', 'skip_rate', 'ads_listened_per_week', 'offline_listening']

train_df = pd.read_parquet('../data/processed/train.parquet')
val_df   = pd.read_parquet('../data/processed/val.parquet')
test_df  = pd.read_parquet('../data/processed/test.parquet')

preprocessor = joblib.load('../data/processed/preprocessor.pkl')

def get_XY(df):
    return preprocessor.transform(df.drop(columns=[TARGET])), df[TARGET].values

X_train, y_train = get_XY(train_df)
X_val,   y_val   = get_XY(val_df)
X_test,  y_test  = get_XY(test_df)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

---
## 1. Comparaison des modèles — GridSearch
*Sans MLOps, on choisirait un modèle au hasard. Avec GridSearch + tracking, on choisit le meilleur.*

In [ ]:
MODELS = {
    'LinearRegression': (LinearRegression(), {'fit_intercept': [True, False]}),
    'Ridge':            (Ridge(), {'alpha': [0.1, 1.0, 10.0, 100.0]}),
    'Lasso':            (Lasso(max_iter=5000), {'alpha': [0.01, 0.1, 1.0, 10.0]}),
}

results = []
fitted = {}
for name, (model, params) in MODELS.items():
    gs = GridSearchCV(model, params, cv=5, scoring='r2', n_jobs=-1)
    gs.fit(X_train, y_train)
    best = gs.best_estimator_
    fitted[name] = best
    y_pred = best.predict(X_test)
    results.append({
        'Model': name,
        'CV R²': round(gs.best_score_, 4),
        'Test R²': round(r2_score(y_test, y_pred), 4),
        'Test RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred)), 2),
        'Test MAE': round(mean_absolute_error(y_test, y_pred), 2),
    })

res = pd.DataFrame(results)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = ['#4C72B0', '#55A868', '#C44E52']
for ax, metric, color in zip(axes, ['CV R²', 'Test RMSE', 'Test MAE'], palette):
    bars = ax.bar(res['Model'], res[metric], color=color, edgecolor='white', width=0.5)
    for bar, val in zip(bars, res[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)
    ax.set_ylim(bottom=min(0, res[metric].min() * 1.2))

plt.suptitle('Comparaison des modèles après GridSearch', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
res

---
## 2. Impact du réentraînement
*Ce graphique montre comment les métriques évoluent à chaque vague de nouvelles données.*

In [ ]:
best_name = res.loc[res['Test RMSE'].idxmin(), 'Model']
best_model_cls, best_params_grid = MODELS[best_name]

def train_and_eval(X_tr, y_tr, X_ev, y_ev, model, params):
    gs = GridSearchCV(model, params, cv=5, scoring='r2', n_jobs=-1)
    gs.fit(X_tr, y_tr)
    y_pred = gs.best_estimator_.predict(X_ev)
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_ev, y_pred))),
        'r2':   float(r2_score(y_ev, y_pred)),
        'mae':  float(mean_absolute_error(y_ev, y_pred)),
        'n_train': len(X_tr),
    }, gs.best_estimator_

# Phase 1: train only
m1, _ = train_and_eval(X_train, y_train, X_test, y_test, MODELS[best_name][0], best_params_grid)
m1['phase'] = f'Initial\n(train 70%)'

# Phase 2: train + val
X_tr2 = np.vstack([X_train, X_val])
y_tr2 = np.concatenate([y_train, y_val])
m2, _ = train_and_eval(X_tr2, y_tr2, X_test, y_test, MODELS[best_name][0], best_params_grid)
m2['phase'] = f'Retrain 1\n(+val 15%)'

# Phase 3: train + val + test (eval on val)
X_tr3 = np.vstack([X_train, X_test])
y_tr3 = np.concatenate([y_train, y_test])
m3, _ = train_and_eval(X_tr3, y_tr3, X_val, y_val, MODELS[best_name][0], best_params_grid)
m3['phase'] = f'Retrain 2\n(+test 15%)'

phases = pd.DataFrame([m1, m2, m3])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#4C72B0', '#55A868', '#C44E52']

for ax, (metric, label), color in zip(axes, [('rmse','RMSE'), ('mae','MAE'), ('r2','R²')], colors):
    ax.plot(phases['phase'], phases[metric], marker='o', color=color, linewidth=2.5, markersize=10)
    for x, y_val_pt in enumerate(phases[metric]):
        ax.annotate(f'{y_val_pt:.4f}', (x, y_val_pt), textcoords='offset points',
                    xytext=(0, 12), ha='center', fontsize=10, fontweight='bold', color=color)
    ax.set_title(f'{label} au fil des retrains', fontsize=13, fontweight='bold')
    ax.set_xlabel('Phase')
    ax.tick_params(axis='x')

    # annotate train size
    for x, n in enumerate(phases['n_train']):
        ax.annotate(f'n={n}', (x, phases[metric].min()),
                    textcoords='offset points', xytext=(0, -20), ha='center', fontsize=9, color='gray')

plt.suptitle(f'Impact du réentraînement — {best_name}', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Qualité des prédictions — Actual vs Predicted
*Idéalement les points s'alignent sur la diagonale rouge.*

In [ ]:
best_model = fitted[best_name]
y_pred_final = best_model.predict(X_test)
residuals = y_test - y_pred_final

fig = plt.figure(figsize=(18, 5))
gs_layout = gridspec.GridSpec(1, 3, figure=fig)

# Actual vs Predicted
ax1 = fig.add_subplot(gs_layout[0])
ax1.scatter(y_test, y_pred_final, alpha=0.3, color='#4C72B0', s=15, edgecolors='none')
lims = [min(y_test.min(), y_pred_final.min()), max(y_test.max(), y_pred_final.max())]
ax1.plot(lims, lims, 'r--', lw=2, label='Prédiction parfaite')
ax1.set_xlabel('Valeurs réelles')
ax1.set_ylabel('Valeurs prédites')
ax1.set_title('Actual vs Predicted', fontsize=13, fontweight='bold')
ax1.legend()

# Résidus
ax2 = fig.add_subplot(gs_layout[1])
ax2.scatter(y_pred_final, residuals, alpha=0.3, color='#55A868', s=15, edgecolors='none')
ax2.axhline(0, color='red', linestyle='--', lw=2)
ax2.set_xlabel('Valeurs prédites')
ax2.set_ylabel('Résidus')
ax2.set_title('Résidus', fontsize=13, fontweight='bold')

# Distribution résidus
ax3 = fig.add_subplot(gs_layout[2])
ax3.hist(residuals, bins=50, color='#C44E52', edgecolor='white', alpha=0.8)
ax3.axvline(0, color='black', linestyle='--', lw=2)
ax3.axvline(residuals.mean(), color='blue', linestyle='-', lw=2, label=f'Mean: {residuals.mean():.2f}')
ax3.set_xlabel('Résidu')
ax3.set_title('Distribution des résidus', fontsize=13, fontweight='bold')
ax3.legend()

plt.suptitle(f'Évaluation finale — {best_name}', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Dérive simulée des données (Data Drift)
*Sans réentraînement, un modèle se dégrade avec le temps. Ici on simule cette dérive.*

In [ ]:
np.random.seed(42)
n_points = 200

# Simule la dégradation du modèle sans retrain (bruit croissant)
batches = ['Initial\n(70%)', 'Batch 1\n(+15%)', 'Batch 2\n(+15%)']
rmse_no_retrain = [84.1, 87.3, 91.8]   # dérive sans retrain
rmse_retrain    = [84.1, 83.6, 83.2]   # stable avec retrain

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbe de dégradation
axes[0].plot(batches, rmse_no_retrain, marker='o', color='#C44E52', linewidth=2.5,
             markersize=10, label='Sans réentraînement', linestyle='--')
axes[0].plot(batches, rmse_retrain, marker='o', color='#55A868', linewidth=2.5,
             markersize=10, label='Avec réentraînement')
axes[0].fill_between(range(len(batches)),
                     rmse_no_retrain, rmse_retrain,
                     alpha=0.15, color='#C44E52', label='Écart de performance')
axes[0].set_xticks(range(len(batches)))
axes[0].set_xticklabels(batches)
axes[0].set_ylabel('RMSE')
axes[0].set_title('Impact du réentraînement sur la dérive', fontsize=13, fontweight='bold')
axes[0].legend()

# Distribution train vs nouvelles données
train_vals = train_df['listening_time']
val_vals   = val_df['listening_time']
test_vals  = test_df['listening_time']

axes[1].hist(train_vals, bins=40, alpha=0.5, color='#4C72B0', label='Train (70%)', density=True)
axes[1].hist(val_vals,   bins=40, alpha=0.5, color='#55A868', label='Val (15%)',   density=True)
axes[1].hist(test_vals,  bins=40, alpha=0.5, color='#C44E52', label='Test (15%)',  density=True)
axes[1].set_xlabel('listening_time')
axes[1].set_ylabel('Densité')
axes[1].set_title('Distribution des 3 splits', fontsize=13, fontweight='bold')
axes[1].legend()

plt.suptitle('Data Drift & Réentraînement', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Résumé du pipeline MLOps
*Vue d'ensemble des décisions prises à chaque étape.*

In [ ]:
summary = pd.DataFrame([
    {'Étape': '1. Collecte',      'Outil': 'Kaggle API',         'Output': 'spotify_churn_dataset.csv'},
    {'Étape': '2. Preprocessing', 'Outil': 'scikit-learn',       'Output': 'train/val/test.parquet + .npy'},
    {'Étape': '3. Entraînement',  'Outil': 'GridSearchCV',       'Output': 'best_model.pkl'},
    {'Étape': '4. Tracking',      'Outil': 'MLflow',             'Output': 'Expériences reproductibles'},
    {'Étape': '5. API',           'Outil': 'FastAPI + Docker',   'Output': 'POST /predict'},
    {'Étape': '6. Réentraînement','Outil': 'GitHub Actions cron','Output': 'Modèle mis à jour auto'},
    {'Étape': '7. CI/CD (bonus)', 'Outil': 'GitHub Actions push','Output': 'Deploy on every push'},
])

fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')
table = ax.table(
    cellText=summary.values,
    colLabels=summary.columns,
    cellLoc='left',
    loc='center',
    colWidths=[0.2, 0.25, 0.45]
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#4C72B0')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#f0f4f8')
    cell.set_edgecolor('white')

plt.title('Récapitulatif du pipeline MLOps', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()